# 🧬 Nucleic Acid Structure: DNA and RNA

---

## Learning Objectives

- Understand DNA and RNA chemical composition
- Learn about A-form and B-form DNA structures
- Analyze structural differences between DNA and RNA
- Work with nucleic acid structures in PDB files

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Nucleotide Components

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        NUCLEOTIDE STRUCTURE                             │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Nucleotide = Phosphate + Sugar + Base                                 │
│                                                                         │
│        O-                                                               │
│        ║                                                                │
│    O=P-O-5'CH2                                                          │
│        ║      ╲    Base                                                 │
│        O-      O                                                        │
│                │   /                                                    │
│   Phosphate   Sugar (Ribose/Deoxyribose)                                │
│                │                                                        │
│               3'OH                                                      │
│                                                                         │
│   DNA Sugar: Deoxyribose (2' position has H)                            │
│   RNA Sugar: Ribose (2' position has OH)                                │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘

Bases:
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│   PURINES (2 rings)           PYRIMIDINES (1 ring)                      │
│   ┌───────────────┐           ┌───────────────┐                         │
│   │ Adenine (A)   │           │ Cytosine (C)  │                         │
│   │ Guanine (G)   │           │ Thymine (T)   │ ← DNA only              │
│   └───────────────┘           │ Uracil (U)    │ ← RNA only              │
│                               └───────────────┘                         │
│                                                                         │
│   Watson-Crick Base Pairing:                                            │
│   A═══T (or A═══U in RNA)  2 hydrogen bonds                             │
│   G≡≡≡C                    3 hydrogen bonds                             │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Nucleotide properties dictionary
NUCLEOTIDES = {
    'DNA': {
        'A': {'name': 'Adenine', 'type': 'purine', 'pairs_with': 'T', 'h_bonds': 2},
        'T': {'name': 'Thymine', 'type': 'pyrimidine', 'pairs_with': 'A', 'h_bonds': 2},
        'G': {'name': 'Guanine', 'type': 'purine', 'pairs_with': 'C', 'h_bonds': 3},
        'C': {'name': 'Cytosine', 'type': 'pyrimidine', 'pairs_with': 'G', 'h_bonds': 3}
    },
    'RNA': {
        'A': {'name': 'Adenine', 'type': 'purine', 'pairs_with': 'U', 'h_bonds': 2},
        'U': {'name': 'Uracil', 'type': 'pyrimidine', 'pairs_with': 'A', 'h_bonds': 2},
        'G': {'name': 'Guanine', 'type': 'purine', 'pairs_with': 'C', 'h_bonds': 3},
        'C': {'name': 'Cytosine', 'type': 'pyrimidine', 'pairs_with': 'G', 'h_bonds': 3}
    }
}

def complement_sequence(seq, nucleic_acid_type='DNA'):
    """
    Generate complement sequence.
    
    Parameters:
        seq: DNA or RNA sequence
        nucleic_acid_type: 'DNA' or 'RNA'
    """
    bases = NUCLEOTIDES[nucleic_acid_type]
    complement = ''
    for base in seq.upper():
        if base in bases:
            complement += bases[base]['pairs_with']
        else:
            complement += 'N'  # Unknown base
    return complement

def reverse_complement(seq, nucleic_acid_type='DNA'):
    """Generate reverse complement (5'→3' convention)."""
    return complement_sequence(seq, nucleic_acid_type)[::-1]

# Example
dna = "ATGCGATCGA"
print(f"DNA sequence:      5'-{dna}-3'")
print(f"Complement:        3'-{complement_sequence(dna)}-5'")
print(f"Reverse complement: 5'-{reverse_complement(dna)}-3'")

---

## A-form vs B-form DNA

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    DNA HELICAL FORMS                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Parameter              A-form          B-form          Z-form          │
│  ─────────────────────────────────────────────────────────────────      │
│  Helix sense           Right-handed    Right-handed    Left-handed      │
│  Base pairs per turn   11              10.5            12               │
│  Rise per bp           2.6 Å           3.4 Å           3.7 Å            │
│  Helix diameter        23 Å            20 Å            18 Å             │
│  Major groove          Narrow/deep     Wide/deep       Flat             │
│  Minor groove          Wide/shallow    Narrow/deep     Narrow/deep      │
│                                                                         │
│  B-form (physiological):        A-form (dehydrated):                    │
│       ╱╲                              ╱╲                                │
│      ╱  ╲                            ╱  ╲                               │
│     ╱    ╲                          ┃    ┃                              │
│    ╱      ╲                         ┃    ┃  ← More compact              │
│   ╱        ╲                        ┃    ┃                              │
│   ╲        ╱                         ╲  ╱                               │
│    ╲      ╱                           ╲╱                                │
│     ╲    ╱                                                              │
│      ╲  ╱                                                               │
│       ╲╱                                                                │
│                                                                         │
│  RNA is typically A-form due to 2'-OH steric constraints                │
│  DNA-RNA hybrids adopt intermediate A/B conformation                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Helical parameters for different DNA forms
HELIX_PARAMS = {
    'A-form': {
        'bp_per_turn': 11,
        'rise_per_bp': 2.6,  # Angstroms
        'diameter': 23,       # Angstroms
        'handedness': 'right',
        'conditions': 'dehydrated, DNA-RNA hybrids'
    },
    'B-form': {
        'bp_per_turn': 10.5,
        'rise_per_bp': 3.4,
        'diameter': 20,
        'handedness': 'right',
        'conditions': 'physiological (92% humidity)'
    },
    'Z-form': {
        'bp_per_turn': 12,
        'rise_per_bp': 3.7,
        'diameter': 18,
        'handedness': 'left',
        'conditions': 'high salt, alternating purine-pyrimidine'
    }
}

def calculate_helix_length(num_bp, form='B-form'):
    """
    Calculate approximate helix length for given base pairs.
    
    Returns length in Angstroms.
    """
    params = HELIX_PARAMS[form]
    return num_bp * params['rise_per_bp']

def calculate_helix_turns(num_bp, form='B-form'):
    """Calculate number of helical turns."""
    params = HELIX_PARAMS[form]
    return num_bp / params['bp_per_turn']

# Example: Human chromosome 1 (~249 million bp)
chr1_bp = 249_000_000
length_A = calculate_helix_length(chr1_bp, 'B-form')
length_cm = length_A * 1e-8  # Convert Angstroms to cm

print(f"Human chromosome 1 ({chr1_bp:,} bp):")
print(f"  B-form length: {length_cm:.1f} cm")
print(f"  Helical turns: {calculate_helix_turns(chr1_bp):.1f}")

---

## DNA-Protein Interactions

```
┌─────────────────────────────────────────────────────────────────────────┐
│                   DNA GROOVE RECOGNITION                                │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│                    MAJOR GROOVE                                         │
│                   ╱          ╲                                          │
│                  ╱ Wider,     ╲                                         │
│    5'───────────╱  more info   ╲───────────3'                           │
│         ╲      │    for TFs    │      ╱                                 │
│          ╲     │               │     ╱                                  │
│           ╲────┼───────────────┼────╱                                   │
│          ╱     │               │     ╲                                  │
│         ╱      │   Narrower    │      ╲                                 │
│    3'───────────╲   groove    ╱───────────5'                            │
│                  ╲           ╱                                          │
│                    MINOR GROOVE                                         │
│                                                                         │
│   Major groove: TF recognition (helix-turn-helix, zinc fingers)         │
│   Minor groove: AT-rich binding (TATA-box, histone H1)                  │
│                                                                         │
│   Recognition patterns:                                                 │
│   ┌────────────────────────────────────────────┐                        │
│   │ Base Pair │ Major Groove │ Minor Groove    │                        │
│   ├───────────┼──────────────┼─────────────────┤                        │
│   │ A-T       │ H-bond acc   │ H-bond acc      │                        │
│   │ T-A       │ CH₃ group    │ H-bond acc      │                        │
│   │ G-C       │ H-bond don+  │ H-bond acc/don  │                        │
│   │ C-G       │ H-bond acc   │ H-bond acc      │                        │
│   └────────────────────────────────────────────┘                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_nucleic_acid_pdb(pdb_lines):
    """
    Parse nucleic acid residues from PDB.
    
    Returns dictionary with chain→residue info.
    """
    DNA_RESIDUES = {'DA', 'DT', 'DG', 'DC', 'DU'}  # Deoxy forms
    RNA_RESIDUES = {'A', 'U', 'G', 'C', 'RA', 'RU', 'RG', 'RC'}
    
    chains = {}
    
    for line in pdb_lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            resname = line[17:20].strip()
            chain = line[21]
            resnum = int(line[22:26])
            
            if resname in DNA_RESIDUES or resname in RNA_RESIDUES:
                if chain not in chains:
                    chains[chain] = {
                        'type': 'DNA' if resname in DNA_RESIDUES else 'RNA',
                        'residues': []
                    }
                
                # Add unique residues
                res_id = (resnum, resname)
                if res_id not in [(r['num'], r['name']) for r in chains[chain]['residues']]:
                    chains[chain]['residues'].append({
                        'num': resnum,
                        'name': resname,
                        'base': resname[-1]  # Last character is the base
                    })
    
    return chains

def extract_sequence(chain_data):
    """Extract sequence from parsed chain data."""
    residues = sorted(chain_data['residues'], key=lambda x: x['num'])
    return ''.join(r['base'] for r in residues)

print("Nucleic acid PDB parser ready.")
print("Usage: chains = parse_nucleic_acid_pdb(pdb_lines)")

---

## RNA Secondary Structure

```
┌─────────────────────────────────────────────────────────────────────────┐
│                   RNA SECONDARY STRUCTURE MOTIFS                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   1. STEM-LOOP (Hairpin)           2. BULGE                             │
│         U C                              A                              │
│        U   G                            │                               │
│       G─────C                         G─C                               │
│       │     │                         A─U                               │
│       C─────G                         │ │ ← Unpaired                    │
│       │     │                         C U                               │
│       A─────U                         G─C                               │
│       5'   3'                         5'3'                              │
│                                                                         │
│   3. INTERNAL LOOP                 4. JUNCTION (Multi-stem)             │
│       G─C                                 G─C                           │
│       │ │                                ╱   ╲                          │
│       A U ← Mismatches                  A     U                         │
│       G A                              G─C   A─U                        │
│       │ │                              │ │   │ │                        │
│       C─G                              C─G   G─C                        │
│                                                                         │
│   5. PSEUDOKNOT                                                         │
│       5'──A─U──G─C──╮                                                   │
│              │    │  │ ← Base-pairs with downstream                     │
│       3'──U─A──C─G──╯    region                                         │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_dot_bracket(structure):
    """
    Parse dot-bracket notation for RNA secondary structure.
    
    Returns list of base pairs (i, j) where i < j.
    
    Notation:
      . = unpaired
      ( = 5' side of base pair
      ) = 3' side of base pair
    """
    pairs = []
    stack = []
    
    for i, char in enumerate(structure):
        if char == '(':
            stack.append(i)
        elif char == ')':
            if stack:
                j = stack.pop()
                pairs.append((j, i))
    
    return sorted(pairs)

def count_structure_elements(structure):
    """
    Count structural elements in dot-bracket notation.
    """
    paired = structure.count('(') + structure.count(')')
    unpaired = structure.count('.')
    
    # Find hairpin loops (unpaired regions between paired)
    import re
    hairpins = len(re.findall(r'\([.]+\)', structure))
    
    return {
        'base_pairs': paired // 2,
        'unpaired': unpaired,
        'hairpin_loops': hairpins,
        'length': len(structure)
    }

# Example: tRNA-like structure
structure = "((((((.....))))))....(((((......))))))"
sequence =  "GCGGAUUUAGCUCAGDDGGGAGAGCGCCAGACUGAAYA"

pairs = parse_dot_bracket(structure)
stats = count_structure_elements(structure)

print(f"Structure: {structure}")
print(f"Base pairs: {stats['base_pairs']}")
print(f"Unpaired: {stats['unpaired']}")
print(f"First 5 pairs: {pairs[:5]}")

---

## 🏋️ Exercises

### Exercise 1: GC Content Calculator
Write a function to calculate GC content and melting temperature.

### Exercise 2: Restriction Sites
Find restriction enzyme recognition sites in a DNA sequence.

### Exercise 3: RNA Folding
Predict simple hairpin structures from sequence.

---

## 📚 Resources

- [Nucleic Acid Database](http://ndbserver.rutgers.edu/)
- [3DNA](http://x3dna.org/) - Nucleic acid structure analysis
- [ViennaRNA](https://www.tbi.univie.ac.at/RNA/) - RNA secondary structure prediction

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/10_Nucleic_Acids/01_DNA_RNA_Structure.ipynb`)
